# Chatterbox Multilingual V3: Custom-Text TTS Web UI

Run **Chatterbox Multilingual V3** in Google Colab and use it through a browser-based [Gradio](https://gradio.app/) interface. Enter your own text, choose a supported language, choose a built-in reference voice or upload/record your own voice reference, adjust generation settings, and generate downloadable WAV audio.

The long-form batch tab can synthesize a manuscript that you have already reviewed and approved. It does not extract or approve the *Buddhist Yoga* PDF, inspect the project's approval records, or change the separate repository workflow's full-book synthesis gate.

## What it provides

- Quick free-form TTS for auditions, plus a separate full-length audiobook batch workflow.
- The language list obtained directly from the loaded Chatterbox V3 model.
- Built-in English and Hindi demo references, or your own uploaded/microphone-recorded reference audio.
- Standard and experimental longer-reference conditioning profiles, plus controls for exaggeration, CFG weight, temperature, repetition penalty, min-p, top-p, and seed.
- WAV playback and download, live logs, resumable long-form jobs, selective chunk rerendering, per-section masters, whole-book assembly, technical QA reports, and delivery bundles.
- The same pinned Chatterbox source revision and verified V3 model-file approach used by the project's existing notebook.

**Runtime:** GPU is strongly recommended. CPU fallback is supported but can be very slow. The selected model files require several gigabytes of download/storage.

**Google Drive is optional:** by default, the notebook does not mount or access Drive. The model and generated files stay in temporary Colab storage, and generated WAVs can be downloaded from Gradio. To keep model snapshots and generated WAVs/metadata across runtime resets, set `USE_GOOGLE_DRIVE = True` in the setup cell and rerun from the top. The first Drive-enabled run will ask you to authorize access.

**Privacy:** the Gradio share URL is reachable over the public internet and this app has no login. Anyone who obtains the URL can use the UI, so do not share it publicly. Your uploaded voice reference is processed in the Colab runtime and is not copied into the repository or the Drive output folder. A normalized temporary copy remains under the Colab VM's `/content` directory until that runtime is terminated or reset. Generated JSON metadata includes the entered text and generation settings, so treat saved outputs as private.

**Not a translator:** choose the language that matches the text you entered. For best voice consistency, use a clean reference recording in that same language. The English/Hindi built-in voices are only convenient defaults; for other languages, upload a suitable reference recording.

**Text limit:** up to 12,000 characters per request. Quick TTS requests are limited to 12,000 characters. For a full manuscript, use the Audiobook Jobs tab: it chunks and checkpoints the book, saves each WAV, and can resume after a Colab reset when Drive persistence is enabled.

## 1. Optional Drive setup and install the pinned runtime

This uses the same Chatterbox upstream commit as the existing pronunciation notebook. Gradio is pinned to **6.31.0**. NumPy follows the pinned Chatterbox package metadata: Python 3.13 and newer use NumPy 2.x, while older Python versions use NumPy 1.x. The Colab runtime's Torch and torchaudio versions are preserved. Package installation progress is shown in the cell output.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import json
import hashlib
import gc
import time
import re
import threading
import uuid
import urllib.request
from datetime import datetime, timezone

# Optional persistence. False means no Google Drive authorization is requested.
# Change to True before running this cell to preserve model files and generated outputs.
USE_GOOGLE_DRIVE = False

from google.colab import drive
if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Chatterbox Playground")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Google Drive persistence: ON")
else:
    DRIVE_ROOT = None
    print("Google Drive persistence: OFF (temporary Colab storage only)")

CHATTERBOX_COMMIT = "5de7a54aa4e5e2baadb0182dde554908b48b85c2"

# Match the NumPy requirement in the pinned Chatterbox pyproject.toml:
# Python 3.13+ requires NumPy 2.x; Python 3.12 and older use NumPy 1.x.
NUMPY_REQUIREMENT = (
    "numpy>=2.0.0" if sys.version_info >= (3, 13) else "numpy<2.0.0"
)
print("Python:", sys.version.split()[0], "| NumPy requirement:", NUMPY_REQUIREMENT)

# Let pip show download/build progress instead of appearing frozen.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        NUMPY_REQUIREMENT, "gradio==6.31.0",
    ],
    check=True,
)

import numpy as np
import torch
import torchaudio

TORCH_BEFORE = torch.__version__
TORCHAUDIO_BEFORE = torchaudio.__version__
print("Colab Python:", sys.version.split()[0])
print("Torch:", TORCH_BEFORE, "| torchaudio:", TORCHAUDIO_BEFORE)
print("NumPy:", np.__version__)

numpy_major = int(np.__version__.split(".")[0])
if sys.version_info >= (3, 13) and numpy_major < 2:
    raise RuntimeError("Python 3.13+ requires NumPy 2.x for this Chatterbox revision. Rerun the setup cell.")
if sys.version_info < (3, 13) and numpy_major >= 2:
    raise RuntimeError("Python 3.12 and older require NumPy 1.x for this Chatterbox revision. Rerun the setup cell.")

# Remove an incompatible PyPI installation if one is already present.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "chatterbox-tts"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

source_spec = f"git+https://github.com/resemble-ai/chatterbox.git@{CHATTERBOX_COMMIT}"
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        "--no-deps", source_spec,
    ],
    check=True,
)

dependency_specs = [
    "librosa==0.11.0",
    "s3tokenizer",
    "transformers==5.2.0",
    "diffusers==0.29.0",
    "git+https://github.com/resemble-ai/Perth.git@master",
    "conformer==0.3.2",
    "safetensors==0.5.3",
    "omegaconf",
    "soundfile",
    "huggingface_hub",
]
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        *dependency_specs,
    ],
    check=True,
)

import librosa
import soundfile as sf
import gradio as gr
from safetensors import safe_open
from huggingface_hub import HfApi, snapshot_download
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

assert torch.__version__ == TORCH_BEFORE, (
    f"Torch changed during installation: {TORCH_BEFORE} -> {torch.__version__}"
)
assert torchaudio.__version__ == TORCHAUDIO_BEFORE, (
    f"torchaudio changed during installation: {TORCHAUDIO_BEFORE} -> {torchaudio.__version__}"
)

print("Gradio:", gr.__version__)
print("Pinned Chatterbox source:", CHATTERBOX_COMMIT)
print("Torch and torchaudio were preserved.")

## 2. Download or reuse the verified Multilingual V3 model files

The full model repository contains files that this notebook does not need. It downloads only the six files used by the existing project workflow, pins the download to a specific Hugging Face model revision, checks their sizes, and saves a verified copy in a dedicated Drive folder.

This model snapshot is separate from the audiobook project's existing model-snapshot folder.

In [ ]:
HF_HOME = Path("/content/chatterbox-playground-hf")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")

MODEL_REPO = "ResembleAI/chatterbox"
model_info = HfApi().model_info(MODEL_REPO, revision="main", files_metadata=True)
MODEL_REVISION = model_info.sha

MODEL_FILES = [
    "ve.pt",
    "s3gen.pt",
    "t3_mtl23ls_v3.safetensors",
    "grapheme_mtl_merged_expanded_v1.json",
    "conds.pt",
    "Cangjie5_TC.json",
]
REMOTE_SIZES = {
    item.rfilename: getattr(item, "size", None)
    for item in model_info.siblings
    if item.rfilename in MODEL_FILES
}
MINIMUM_SIZES = {
    "ve.pt": 5_000_000,
    "s3gen.pt": 1_000_000_000,
    "t3_mtl23ls_v3.safetensors": 2_000_000_000,
    "grapheme_mtl_merged_expanded_v1.json": 60_000,
    "conds.pt": 50_000,
    "Cangjie5_TC.json": 1_000_000,
}

DRIVE_MODEL_DIR = (
    DRIVE_ROOT / "model-snapshots" / f"chatterbox-v3-{MODEL_REVISION[:12]}"
    if USE_GOOGLE_DRIVE else None
)
LOCAL_MODEL_DIR = Path("/content/chatterbox-playground-model")


def check_snapshot(folder: Path) -> list[str]:
    problems = []
    for filename in MODEL_FILES:
        path = folder / filename
        if not path.is_file():
            problems.append(f"{filename}: missing")
            continue

        actual_size = path.stat().st_size
        expected_size = REMOTE_SIZES.get(filename)
        if expected_size and actual_size != expected_size:
            problems.append(
                f"{filename}: {actual_size:,} bytes; Hub reports {expected_size:,}"
            )
        elif not expected_size and actual_size < MINIMUM_SIZES[filename]:
            problems.append(f"{filename}: suspiciously small ({actual_size:,} bytes)")
    return problems


if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE and not check_snapshot(DRIVE_MODEL_DIR):
    print("Reusing the verified Multilingual V3 snapshot from Drive.")
    for filename in MODEL_FILES:
        shutil.copy2(DRIVE_MODEL_DIR / filename, LOCAL_MODEL_DIR / filename)
else:
    if USE_GOOGLE_DRIVE and DRIVE_MODEL_DIR.exists():
        print("The dedicated Drive snapshot is incomplete or mismatched; replacing that snapshot only.")
        shutil.rmtree(DRIVE_MODEL_DIR)

    print("Downloading the selected V3 model files. This is several gigabytes...")
    snapshot_download(
        repo_id=MODEL_REPO,
        repo_type="model",
        revision=MODEL_REVISION,
        allow_patterns=MODEL_FILES,
        local_dir=str(LOCAL_MODEL_DIR),
        force_download=True,
    )

    problems = check_snapshot(LOCAL_MODEL_DIR)
    if problems:
        raise RuntimeError("Model snapshot verification failed:\n" + "\n".join(problems))

    if USE_GOOGLE_DRIVE:
        try:
            DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            for filename in MODEL_FILES:
                shutil.copy2(LOCAL_MODEL_DIR / filename, DRIVE_MODEL_DIR / filename)

            (DRIVE_MODEL_DIR / "snapshot_manifest.json").write_text(
                json.dumps(
                    {
                        "repository": MODEL_REPO,
                        "revision": MODEL_REVISION,
                        "files": {
                            filename: {
                                "size_bytes": (DRIVE_MODEL_DIR / filename).stat().st_size
                            }
                            for filename in MODEL_FILES
                        },
                    },
                    indent=2,
                ),
                encoding="utf-8",
            )
            print("Verified model files saved to Drive.")
        except Exception as exc:
            print("Could not persist the model snapshot to Drive:", type(exc).__name__, str(exc))
            print("The verified local snapshot is still available for this runtime.")
    else:
        print("Model snapshot is in temporary Colab storage and will need downloading again after a runtime reset.")

problems = check_snapshot(LOCAL_MODEL_DIR)
if problems:
    raise RuntimeError("Local model snapshot is incomplete:\n" + "\n".join(problems))

print("Model revision:", MODEL_REVISION)
for filename in MODEL_FILES:
    print(f"{filename}: {(LOCAL_MODEL_DIR / filename).stat().st_size / 1024**2:.1f} MiB")

## 3. Preflight and load Chatterbox Multilingual V3

This checks the two PyTorch checkpoints with safe `weights_only=True` loading, checks the safetensors header, preserves the runtime's Torch versions, and loads the explicit V3 model selector. The notebook falls back to CPU if Colab did not provide a GPU.

In [ ]:
def safe_checkpoint_preflight(filename: str, force_refresh: bool = False) -> None:
    path = LOCAL_MODEL_DIR / filename
    try:
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )
        if not isinstance(state, dict) or not state:
            raise RuntimeError(f"{filename} did not contain a non-empty state dictionary.")
        print(f"{filename}: safe checkpoint read ({len(state)} top-level entries)")
        del state
        gc.collect()
    except Exception as exc:
        if force_refresh:
            raise RuntimeError(
                f"{filename} still fails after a fresh Hub download: {type(exc).__name__}: {exc}"
            ) from exc
        print(f"{filename} failed preflight ({type(exc).__name__}: {exc}). Refreshing this file only.")
        snapshot_download(
            repo_id=MODEL_REPO,
            repo_type="model",
            revision=MODEL_REVISION,
            allow_patterns=[filename],
            local_dir=str(LOCAL_MODEL_DIR),
            force_download=True,
        )
        safe_checkpoint_preflight(filename, force_refresh=True)
        if USE_GOOGLE_DRIVE and DRIVE_MODEL_DIR.exists():
            shutil.copy2(path, DRIVE_MODEL_DIR / filename)


safe_checkpoint_preflight("ve.pt")
safe_checkpoint_preflight("s3gen.pt")

with safe_open(
    str(LOCAL_MODEL_DIR / "t3_mtl23ls_v3.safetensors"),
    framework="pt",
    device="cpu",
) as handle:
    if not list(handle.keys()):
        raise RuntimeError("The V3 T3 safetensors file is empty.")
    print("V3 T3 safetensors header: valid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory / 1024**3, 1))
else:
    print("No GPU detected. Loading on CPU; generation may be very slow.")

print(f"Loading Chatterbox Multilingual V3 on {DEVICE}...")
load_started = time.perf_counter()
model = ChatterboxMultilingualTTS.from_local(
    str(LOCAL_MODEL_DIR),
    device=DEVICE,
    t3_model="v3",
)
load_seconds = time.perf_counter() - load_started

assert int(model.sr) == 24000, f"Expected 24 kHz output, got {model.sr}."
SUPPORTED_LANGUAGES = ChatterboxMultilingualTTS.get_supported_languages()
assert isinstance(SUPPORTED_LANGUAGES, dict) and "en" in SUPPORTED_LANGUAGES

LANGUAGE_CHOICES = [
    (f"{language_name} ({language_code})", language_code)
    for language_code, language_name in sorted(SUPPORTED_LANGUAGES.items())
]

print(f"Loaded in {load_seconds:.1f} seconds.")
print(f"Device: {DEVICE} | Output sample rate: {model.sr} Hz")
print("Supported language IDs:", ", ".join(sorted(SUPPORTED_LANGUAGES)))
print("The language dropdown in the next cell is populated from this model.")

## 4. Prepare reference voices, output folders, and synthesis helpers

A reference clip is used for voice conditioning, not as text to be transcribed. The standard profile uses the model's original 6-second T3 prompt and 10-second S3Gen acoustic prompt. Two experimental profiles can use a longer speaker sample and/or extend those prompt windows. Longer windows are not the upstream-trained defaults, so compare their audio against the standard profile rather than assuming longer is better.

Your own recording is normalized to mono 24 kHz PCM. Experimental profiles can retain up to 30 seconds for the speaker-identity encoder; the reference remains temporary in the Colab runtime and is not copied to Drive.

In [ ]:
REFERENCE_DIR = Path("/content/chatterbox-playground/references")
LOCAL_OUTPUT_DIR = Path("/content/chatterbox-playground/outputs")
DRIVE_OUTPUT_DIR = (DRIVE_ROOT / "generated") if USE_GOOGLE_DRIVE else None
for folder in (REFERENCE_DIR, LOCAL_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if DRIVE_OUTPUT_DIR is not None:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Three inference-time reference profiles. The first is the upstream default.
# The longer profiles are experiments, not upstream-supported/training defaults.
CONDITIONING_PROFILES = {
    "Standard (T3 6s / S3Gen 10s)": {
        "t3_prompt_seconds": 6,
        "s3gen_prompt_seconds": 10,
        "voice_encoder_max_seconds": 10,
        "experimental": False,
    },
    "Longer VoiceEncoder sample (T3 6s / S3Gen 10s / VoiceEncoder up to 30s)": {
        "t3_prompt_seconds": 6,
        "s3gen_prompt_seconds": 10,
        "voice_encoder_max_seconds": 30,
        "experimental": True,
    },
    "Extended prompts (T3 10s / S3Gen 15s / VoiceEncoder up to 30s)": {
        "t3_prompt_seconds": 10,
        "s3gen_prompt_seconds": 15,
        "voice_encoder_max_seconds": 30,
        "experimental": True,
    },
}
T3_REFERENCE_SAMPLE_RATE = 16_000
SPEECH_TOKEN_RATE = 25
MAX_REFERENCE_AUDIO_SECONDS = 30

DEMO_REFERENCE_URLS = {
    "english": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac",
    "hindi": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/hi_f1.flac",
}
DEMO_REFERENCE_FILES = {
    language: REFERENCE_DIR / f"official_demo_{language}.flac"
    for language in DEMO_REFERENCE_URLS
}

for language, url in DEMO_REFERENCE_URLS.items():
    path = DEMO_REFERENCE_FILES[language]
    if not path.is_file() or path.stat().st_size < 1024:
        print(f"Downloading built-in {language} reference...")
        urllib.request.urlretrieve(url, path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalize_reference_audio(
    source_path: str | Path,
    max_duration_seconds: int = 10,
) -> tuple[Path, dict]:
    """Normalize a reference to mono 24 kHz PCM, retaining the requested leading window."""
    if not 1 <= int(max_duration_seconds) <= MAX_REFERENCE_AUDIO_SECONDS:
        raise ValueError(f"Reference duration must be between 1 and {MAX_REFERENCE_AUDIO_SECONDS} seconds.")
    source = Path(source_path)
    if not source.is_file():
        raise FileNotFoundError(f"Reference audio not found: {source}")

    audio, sample_rate = librosa.load(str(source), sr=24000, mono=True)
    if audio.ndim != 1 or audio.size == 0 or not np.isfinite(audio).all():
        raise ValueError("The reference audio is empty or invalid.")
    original_seconds = audio.size / 24000
    if original_seconds < 1.0:
        raise ValueError("Reference audio must contain at least one second of usable audio.")

    audio = audio[: int(max_duration_seconds) * 24000]
    pad = (-audio.size) % 960
    if pad:
        audio = np.pad(audio, (0, pad))

    digest = hashlib.sha256(audio.tobytes()).hexdigest()
    normalized_path = REFERENCE_DIR / f"normalized_{digest[:16]}.wav"
    if not normalized_path.is_file():
        sf.write(str(normalized_path), audio, 24000, subtype="PCM_16")

    return normalized_path, {
        "source_name": source.name,
        "source_sha256": sha256_file(source),
        "normalized_sha256": sha256_file(normalized_path),
        "original_duration_seconds": round(original_seconds, 3),
        "conditioning_duration_seconds": round(audio.size / 24000, 3),
        "sample_rate": 24000,
        "maximum_reference_duration_seconds": int(max_duration_seconds),
    }


def split_text_into_chunks(text: str, max_chars: int = 420) -> list[str]:
    """Pack sentences into conservative chunks; split very long sentences at word boundaries."""
    cleaned = text.strip()
    if not cleaned:
        raise ValueError("Enter some text first.")
    if len(cleaned) > MAX_TEXT_CHARACTERS:
        raise ValueError(f"Text is longer than the {MAX_TEXT_CHARACTERS:,}-character limit.")

    paragraphs = [part.strip() for part in cleaned.splitlines() if part.strip()]
    if not paragraphs:
        raise ValueError("Enter some text first.")

    chunks = []
    for paragraph in paragraphs:
        sentences = re.split(r"(?<=[.!?])\s+", paragraph)
        current = ""

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            if len(sentence) > max_chars:
                if current:
                    chunks.append(current)
                    current = ""
                words = sentence.split()
                piece = ""
                for word in words:
                    if len(word) > max_chars:
                        raise ValueError(
                            f"A single word/token is longer than the {max_chars}-character chunk limit."
                        )
                    candidate = word if not piece else piece + " " + word
                    if len(candidate) <= max_chars:
                        piece = candidate
                    else:
                        chunks.append(piece)
                        piece = word
                if piece:
                    chunks.append(piece)
                continue

            candidate = sentence if not current else current + " " + sentence
            if len(candidate) <= max_chars:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                current = sentence

        if current:
            chunks.append(current)

    if not chunks:
        raise ValueError("No usable text remained after splitting.")
    return chunks


MODEL_LOCK = threading.Lock()
INTER_CHUNK_PAUSE_SECONDS = 0.18
MAX_TEXT_CHARACTERS = 12000
MAX_CHUNK_CHARACTERS = 420

print("Reference voices cached.")
print("Local temporary output:", LOCAL_OUTPUT_DIR)
print("Drive output:", DRIVE_OUTPUT_DIR if DRIVE_OUTPUT_DIR is not None else "disabled")
print("Sentence-aware chunker ready.")

## 5. Build the text-to-speech function

Each request prepares the selected reference voice, generates the text chunks in order, inserts a short pause between chunks, writes a 24 kHz PCM WAV, and stores a metadata sidecar. A modest peak ceiling prevents accidental full-scale overshoots when the pieces are joined.

Only one generation request runs at a time because the model keeps voice-conditioning state in memory.

In [ ]:
def audit_text_tokenization(chunks: list[str], language_id: str) -> dict:
    """Count tokenizer [UNK] tokens before synthesis; never rewrites the user's text."""
    backend = getattr(getattr(model, "tokenizer", None), "tokenizer", None)
    if backend is None or not hasattr(backend, "token_to_id"):
        return {
            "available": False,
            "unknown_token_count": None,
            "chunks": [],
        }

    unknown_id = backend.token_to_id("[UNK]")
    if unknown_id is None:
        return {
            "available": False,
            "unknown_token_count": None,
            "chunks": [],
        }

    chunk_audits = []
    for index, chunk in enumerate(chunks, start=1):
        token_tensor = model.tokenizer.text_to_tokens(chunk, language_id=language_id)
        token_ids = token_tensor.reshape(-1).tolist()
        unknown_count = sum(int(token_id) == int(unknown_id) for token_id in token_ids)
        chunk_audits.append({
            "chunk_index": index,
            "token_count": len(token_ids),
            "unknown_token_count": unknown_count,
        })

    return {
        "available": True,
        "unknown_token_count": sum(item["unknown_token_count"] for item in chunk_audits),
        "chunks": chunk_audits,
    }


def generate_speech(
    text: str,
    language_id: str,
    reference_mode: str,
    uploaded_reference: str | None,
    conditioning_profile: str,
    exaggeration: float,
    cfg_weight: float,
    temperature: float,
    repetition_penalty: float,
    min_p: float,
    top_p: float,
    seed_value: float,
    progress=gr.Progress(),
):
    started = time.perf_counter()
    append_app_log(
        "quick_generation_started",
        "Started a quick TTS request.",
        details={"language_id": language_id, "conditioning_profile": conditioning_profile},
    )

    try:
        chunks = split_text_into_chunks(text, max_chars=MAX_CHUNK_CHARACTERS)

        if conditioning_profile not in CONDITIONING_PROFILES:
            raise ValueError(f"Unknown conditioning profile: {conditioning_profile}")
        conditioning_settings = CONDITIONING_PROFILES[conditioning_profile]

        if language_id not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language ID: {language_id}")

        text_token_audit = audit_text_tokenization(chunks, language_id)

        if reference_mode == "Upload/record my own voice":
            if not uploaded_reference:
                raise ValueError("Upload or record a reference audio clip first.")
            reference_source = Path(uploaded_reference)
            reference_label = "user-uploaded reference"
        elif reference_mode == "Built-in English demo voice":
            reference_source = DEMO_REFERENCE_FILES["english"]
            reference_label = "Chatterbox English demo"
        elif reference_mode == "Built-in Hindi demo voice":
            reference_source = DEMO_REFERENCE_FILES["hindi"]
            reference_label = "Chatterbox Hindi demo"
        elif reference_mode == "Automatic (English/Hindi demo; English fallback)":
            if language_id == "hi":
                reference_source = DEMO_REFERENCE_FILES["hindi"]
                reference_label = "automatic Hindi demo"
            else:
                reference_source = DEMO_REFERENCE_FILES["english"]
                reference_label = "automatic English demo"
        else:
            raise ValueError(f"Unknown reference mode: {reference_mode}")

        normalized_reference, reference_info = normalize_reference_audio(
            reference_source,
            max_duration_seconds=conditioning_settings["voice_encoder_max_seconds"],
        )

        request_seed = int(round(seed_value))
        if request_seed < 0:
            import secrets
            request_seed = secrets.randbelow(2**31 - 1)

        generation_parameters = {
            "exaggeration": float(exaggeration),
            "cfg_weight": float(cfg_weight),
            "temperature": float(temperature),
            "repetition_penalty": float(repetition_penalty),
            "min_p": float(min_p),
            "top_p": float(top_p),
        }

        progress(0, desc="Preparing reference voice...")
        pieces = []
        chunk_records = []

        with MODEL_LOCK:
            # Chatterbox's upstream defaults are T3=6s and S3Gen=10s.
            # The extended profile temporarily raises those per-instance
            # limits. The Perceiver and S3Gen accept variable sequence lengths,
            # but longer-than-training prompts remain experimental.
            original_conditioning_limits = (
                model.ENC_COND_LEN,
                model.DEC_COND_LEN,
                model.t3.hp.speech_cond_prompt_len,
            )
            try:
                model.ENC_COND_LEN = int(
                    conditioning_settings["t3_prompt_seconds"] * T3_REFERENCE_SAMPLE_RATE
                )
                model.DEC_COND_LEN = int(
                    conditioning_settings["s3gen_prompt_seconds"] * int(model.sr)
                )
                model.t3.hp.speech_cond_prompt_len = int(
                    conditioning_settings["t3_prompt_seconds"] * SPEECH_TOKEN_RATE
                )
                model.prepare_conditionals(
                    str(normalized_reference),
                    exaggeration=float(exaggeration),
                )
            finally:
                (
                    model.ENC_COND_LEN,
                    model.DEC_COND_LEN,
                    model.t3.hp.speech_cond_prompt_len,
                ) = original_conditioning_limits

            for index, chunk in enumerate(chunks):
                chunk_seed = request_seed + index
                torch.manual_seed(chunk_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed(chunk_seed)
                    torch.cuda.manual_seed_all(chunk_seed)

                progress(
                    (index + 1) / len(chunks),
                    desc=f"Generating chunk {index + 1}/{len(chunks)}",
                )
                generated = model.generate(
                    chunk,
                    language_id=language_id,
                    **generation_parameters,
                )
                waveform = generated.detach().float().cpu().numpy().squeeze()

                if waveform.ndim != 1 or waveform.size == 0 or not np.isfinite(waveform).all():
                    raise RuntimeError(f"Model returned an invalid waveform for chunk {index + 1}.")

                pieces.append(waveform.astype(np.float32, copy=False))
                chunk_records.append({
                    "chunk_index": index + 1,
                    "text": chunk,
                    "duration_seconds": round(waveform.size / int(model.sr), 3),
                    "seed": chunk_seed,
                })

                if index + 1 < len(chunks):
                    pieces.append(
                        np.zeros(
                            int(int(model.sr) * INTER_CHUNK_PAUSE_SECONDS),
                            dtype=np.float32,
                        )
                    )

        combined = np.concatenate(pieces).astype(np.float32, copy=False)
        peak_before_limiter = float(np.max(np.abs(combined)))
        limiter_gain = 1.0
        if peak_before_limiter > 0.98:
            limiter_gain = 0.98 / peak_before_limiter
            combined *= limiter_gain

        request_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
        local_wav = LOCAL_OUTPUT_DIR / f"chatterbox_{request_id}.wav"
        local_metadata = LOCAL_OUTPUT_DIR / f"chatterbox_{request_id}.json"

        sf.write(str(local_wav), combined, int(model.sr), format="WAV", subtype="PCM_16")

        metadata = {
            "request_id": request_id,
            "model": "ResembleAI/chatterbox Multilingual V3",
            "model_revision": MODEL_REVISION,
            "chatterbox_source_commit": CHATTERBOX_COMMIT,
            "device": DEVICE,
            "language_id": language_id,
            "language_name": SUPPORTED_LANGUAGES[language_id],
            "reference_label": reference_label,
            "reference": reference_info,
            "conditioning_profile": conditioning_profile,
            "reference_conditioning": {
                "t3_prompt_seconds": conditioning_settings["t3_prompt_seconds"],
                "s3gen_prompt_seconds": conditioning_settings["s3gen_prompt_seconds"],
                "voice_encoder_reference_seconds": reference_info["conditioning_duration_seconds"],
                "experimental": conditioning_settings["experimental"],
            },
            "text_token_audit": text_token_audit,
            "generation_parameters": generation_parameters,
            "seed": request_seed,
            "chunk_character_limit": MAX_CHUNK_CHARACTERS,
            "inter_chunk_pause_seconds": INTER_CHUNK_PAUSE_SECONDS,
            "chunks": chunk_records,
            "audio": {
                "sample_rate": int(model.sr),
                "duration_seconds": round(combined.size / int(model.sr), 3),
                "peak_before_limiter": round(peak_before_limiter, 6),
                "limiter_gain": round(limiter_gain, 6),
                "sha256": sha256_file(local_wav),
            },
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        local_metadata.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )

        # Drive persistence is optional. The original user reference recording
        # is never copied to Drive.
        if USE_GOOGLE_DRIVE and DRIVE_OUTPUT_DIR is not None:
            drive_wav = DRIVE_OUTPUT_DIR / local_wav.name
            drive_metadata = DRIVE_OUTPUT_DIR / local_metadata.name
            shutil.copy2(local_wav, drive_wav)
            shutil.copy2(local_metadata, drive_metadata)
            saved_location = f"Drive: {drive_wav}"
        else:
            saved_location = f"temporary Colab storage: {local_wav}"

        elapsed = time.perf_counter() - started
        token_warning = ""
        if text_token_audit.get("available") and text_token_audit["unknown_token_count"]:
            token_warning = (
                f" WARNING: tokenizer produced {text_token_audit['unknown_token_count']} "
                "unknown [UNK] text token(s). Check for characters/scripts that do not "
                "match the selected language; use a transliteration or render that term separately."
            )

        status = (
            f"Generated {len(chunks)} chunk(s) in {elapsed:.1f}s on {DEVICE}. "
            f"Duration: {combined.size / int(model.sr):.1f}s. "
            f"Reference: {reference_label}. "
            f"Conditioning: T3 {conditioning_settings['t3_prompt_seconds']}s / "
            f"S3Gen {conditioning_settings['s3gen_prompt_seconds']}s / "
            f"VoiceEncoder {reference_info['conditioning_duration_seconds']:.1f}s. "
            f"{'Experimental profile. ' if conditioning_settings['experimental'] else ''}"
            f"Saved to {saved_location}. Download it before the runtime ends."
            f"{token_warning}"
        )
        append_app_log(
            "quick_generation_complete",
            "Quick TTS request completed.",
            details={"request_id": request_id, "chunks": len(chunks), "duration_seconds": round(combined.size / int(model.sr), 3)},
        )
        return str(local_wav), str(local_wav), status

    except Exception as exc:
        import traceback
        traceback.print_exc()
        append_app_log(
            "quick_generation_failed",
            f"{type(exc).__name__}: {exc}",
            level="ERROR",
        )
        return None, None, f"Generation failed: {type(exc).__name__}: {exc}"


print("Generation function ready.")
print("Maximum text:", MAX_TEXT_CHARACTERS, "characters.")
print("Chunk size:", MAX_CHUNK_CHARACTERS, "characters.")

## 6. Long-form audiobook jobs: preview, generate, resume, and assemble

This adds a separate batch workflow for an already reviewed/approved UTF-8 manuscript. It detects common section headings, creates paragraph-aware chunks, stores each chunk's exact text, seed, settings, output hash, and status, and checkpoints after every successful chunk. Failed jobs can resume without regenerating verified chunks. It creates individual section WAVs, a streamed whole-book master WAV, a JSON manifest, readable/JSONL logs, and a ZIP delivery bundle.

**Important:** this tool does not extract or approve a PDF, compare against the project's approved manuscript tracks, or change the original repository's full-book approval gate. Use the approved text exported by the document workflow. The explicit confirmation in the UI is an additional per-job safeguard, not a replacement for manuscript review.

In [ ]:
# Long-form job support is loaded from an immutable repository commit so the
# planning/checkpoint/assembly helpers are independently testable in the repo.
import importlib.util

AUDIOBOOK_HELPER_COMMIT = "be256652e7ed45648d98be23484150c35bef5f25"
AUDIOBOOK_HELPER_URL = (
    "https://raw.githubusercontent.com/privacyfreak84/colabchatterLLMthingstuff/"
    f"{AUDIOBOOK_HELPER_COMMIT}/tools/longform_audiobook.py"
)
AUDIOBOOK_HELPER_DIR = Path("/content/chatterbox-playground/helpers")
AUDIOBOOK_HELPER_DIR.mkdir(parents=True, exist_ok=True)
AUDIOBOOK_HELPER_PATH = AUDIOBOOK_HELPER_DIR / "longform_audiobook.py"

with urllib.request.urlopen(AUDIOBOOK_HELPER_URL, timeout=30) as response:
    helper_bytes = response.read()
if len(helper_bytes) < 10_000:
    raise RuntimeError("The long-form helper download was incomplete.")
AUDIOBOOK_HELPER_PATH.write_bytes(helper_bytes)
helper_spec = importlib.util.spec_from_file_location(
    "chatterbox_longform_audiobook", AUDIOBOOK_HELPER_PATH
)
if helper_spec is None or helper_spec.loader is None:
    raise RuntimeError("Could not load the long-form audiobook helper.")
longform = importlib.util.module_from_spec(helper_spec)
sys.modules[helper_spec.name] = longform
helper_spec.loader.exec_module(longform)

JOBS_ROOT = (
    DRIVE_OUTPUT_DIR / "audiobook_jobs"
    if DRIVE_OUTPUT_DIR is not None
    else LOCAL_OUTPUT_DIR / "audiobook_jobs"
)
JOBS_ROOT.mkdir(parents=True, exist_ok=True)
APP_LOG_PATH = (
    DRIVE_OUTPUT_DIR / "app_events.jsonl"
    if DRIVE_OUTPUT_DIR is not None
    else LOCAL_OUTPUT_DIR / "app_events.jsonl"
)
APP_LOG_LOCK = threading.Lock()
MAX_BATCH_FILE_BYTES = 20 * 1024 * 1024

def append_app_log(event: str, message: str, details: dict | None = None, level: str = "INFO") -> None:
    """Append a small privacy-conscious app event; never log manuscript text or audio."""
    APP_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    entry = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "level": level.upper(),
        "event": event,
        "message": message,
        "details": details or {},
    }
    with APP_LOG_LOCK:
        with APP_LOG_PATH.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(entry, ensure_ascii=False) + "\n")
            handle.flush()

def read_audiobook_source(uploaded_file: str | None, pasted_text: str | None) -> tuple[str, str]:
    pasted = pasted_text or ""
    if uploaded_file and pasted.strip():
        raise ValueError("Use either an uploaded .txt/.md manuscript or pasted text, not both.")
    if uploaded_file:
        path = Path(uploaded_file)
        if not path.is_file():
            raise FileNotFoundError("The uploaded manuscript file is no longer available.")
        if path.stat().st_size > MAX_BATCH_FILE_BYTES:
            raise ValueError(f"Manuscript file exceeds {MAX_BATCH_FILE_BYTES // (1024 * 1024)} MiB.")
        try:
            text = path.read_text(encoding="utf-8-sig")
        except UnicodeDecodeError as exc:
            raise ValueError("Manuscript must be UTF-8 text (.txt or .md).") from exc
        return text, path.name
    if pasted.strip():
        return pasted, "pasted_manuscript.txt"
    raise ValueError("Upload a UTF-8 .txt/.md manuscript or paste the approved text.")

def preview_audiobook_plan(
    uploaded_file: str | None,
    pasted_text: str | None,
    max_chars: float,
    speak_headings: bool,
    chunk_pause: float,
    paragraph_pause: float,
    section_pause: float,
) -> str:
    try:
        source_text, source_name = read_audiobook_source(uploaded_file, pasted_text)
        plan = longform.plan_audiobook_chunks(
            source_text,
            max_chars=int(max_chars),
            speak_headings=bool(speak_headings),
            chunk_pause_seconds=float(chunk_pause),
            paragraph_pause_seconds=float(paragraph_pause),
            section_pause_seconds=float(section_pause),
        )
        lines = [
            f"Source: {source_name}",
            f"Characters: {plan['source_characters']:,} | Lines: {plan['source_lines']:,}",
            f"Sections: {plan['section_count']} | Chunks: {plan['chunk_count']}",
            f"Maximum chunk size: {plan['max_chunk_characters']} characters",
            f"SHA-256: {plan['source_sha256']}",
            "",
            "SECTION PLAN",
        ]
        for section in plan["sections"]:
            lines.append(f"{section['section_id']}  {section['title']}  ({len(section['chunk_ids'])} chunks)")
            for chunk_id in section["chunk_ids"][:2]:
                chunk = next(item for item in plan["chunks"] if item["chunk_id"] == chunk_id)
                excerpt = chunk["text"].replace("\n", " ")
                if len(excerpt) > 150:
                    excerpt = excerpt[:147] + "..."
                lines.append(f"    {chunk_id}: {excerpt}")
            if len(section["chunk_ids"]) > 2:
                lines.append(f"    ... {len(section['chunk_ids']) - 2} more chunk(s)")
        lines.extend([
            "",
            "No audio has been generated. Review the section boundaries and sample chunk text before starting.",
            "The complete original source will be saved separately; the chunk text is the normalized TTS input.",
        ])
        return "\n".join(lines)
    except Exception as exc:
        return f"Plan error: {type(exc).__name__}: {exc}"

def _resolve_batch_reference(reference_mode: str, uploaded_reference: str | None, language_id: str):
    if reference_mode == "Upload/record my own voice":
        if not uploaded_reference:
            raise ValueError("Upload or record the narrator reference first.")
        source = Path(uploaded_reference)
        label = "user-uploaded reference"
    elif reference_mode == "Built-in English demo voice":
        source = DEMO_REFERENCE_FILES["english"]
        label = "Chatterbox English demo"
    elif reference_mode == "Built-in Hindi demo voice":
        source = DEMO_REFERENCE_FILES["hindi"]
        label = "Chatterbox Hindi demo"
    elif reference_mode == "Automatic (English/Hindi demo; English fallback)":
        if language_id == "hi":
            source = DEMO_REFERENCE_FILES["hindi"]
            label = "automatic Hindi demo"
        else:
            source = DEMO_REFERENCE_FILES["english"]
            label = "automatic English demo"
    else:
        raise ValueError(f"Unknown reference mode: {reference_mode}")
    return source, label

def _prepare_batch_conditionals(normalized_reference: Path, settings: dict, exaggeration: float) -> None:
    original_limits = (
        model.ENC_COND_LEN,
        model.DEC_COND_LEN,
        model.t3.hp.speech_cond_prompt_len,
    )
    try:
        model.ENC_COND_LEN = int(settings["t3_prompt_seconds"] * T3_REFERENCE_SAMPLE_RATE)
        model.DEC_COND_LEN = int(settings["s3gen_prompt_seconds"] * int(model.sr))
        model.t3.hp.speech_cond_prompt_len = int(
            settings["t3_prompt_seconds"] * SPEECH_TOKEN_RATE
        )
        model.prepare_conditionals(
            str(normalized_reference),
            exaggeration=float(exaggeration),
        )
    finally:
        (
            model.ENC_COND_LEN,
            model.DEC_COND_LEN,
            model.t3.hp.speech_cond_prompt_len,
        ) = original_limits

def _batch_log_view(job_dir: Path) -> str:
    return longform.read_log_tail(job_dir / "job.log", max_lines=140)

def run_audiobook_job(
    uploaded_manuscript: str | None,
    pasted_manuscript: str | None,
    requested_job_id: str,
    rerender_chunk_ids: str,
    reroll_seed: bool,
    speak_headings: bool,
    max_chunk_chars: float,
    chunk_pause: float,
    paragraph_pause: float,
    section_pause: float,
    approval_confirmed: bool,
    language_id: str,
    reference_mode: str,
    uploaded_reference: str | None,
    conditioning_profile: str,
    exaggeration: float,
    cfg_weight: float,
    temperature: float,
    repetition_penalty: float,
    min_p: float,
    top_p: float,
    base_seed: float,
    progress=gr.Progress(track_tqdm=False),
):
    job_dir = None
    manifest = None
    try:
        if not approval_confirmed:
            raise ValueError(
                "Confirm that this is the reviewed, approved reading copy and that you authorize this audiobook batch."
            )
        source_text, source_name = read_audiobook_source(
            uploaded_manuscript, pasted_manuscript
        )
        if language_id not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language ID: {language_id}")
        if conditioning_profile not in CONDITIONING_PROFILES:
            raise ValueError(f"Unknown conditioning profile: {conditioning_profile}")
        settings = CONDITIONING_PROFILES[conditioning_profile]
        reference_source, reference_label = _resolve_batch_reference(
            reference_mode, uploaded_reference, language_id
        )
        normalized_reference, reference_info = normalize_reference_audio(
            reference_source,
            max_duration_seconds=settings["voice_encoder_max_seconds"],
        )
        base_seed_value = int(round(base_seed))
        plan = longform.plan_audiobook_chunks(
            source_text,
            max_chars=int(max_chunk_chars),
            speak_headings=bool(speak_headings),
            chunk_pause_seconds=float(chunk_pause),
            paragraph_pause_seconds=float(paragraph_pause),
            section_pause_seconds=float(section_pause),
        )
        generation_parameters = {
            "exaggeration": float(exaggeration),
            "cfg_weight": float(cfg_weight),
            "temperature": float(temperature),
            "repetition_penalty": float(repetition_penalty),
            "min_p": float(min_p),
            "top_p": float(top_p),
        }
        generation_config = {
            "model": "ResembleAI/chatterbox Multilingual V3",
            "model_revision": MODEL_REVISION,
            "chatterbox_source_commit": CHATTERBOX_COMMIT,
            "helper_commit": AUDIOBOOK_HELPER_COMMIT,
            "language_id": language_id,
            "language_name": SUPPORTED_LANGUAGES[language_id],
            "reference_label": reference_label,
            "reference_source_sha256": reference_info["source_sha256"],
            "reference_normalized_sha256": reference_info["normalized_sha256"],
            "reference_original_duration_seconds": reference_info["original_duration_seconds"],
            "reference_conditioning": {
                "t3_prompt_seconds": settings["t3_prompt_seconds"],
                "s3gen_prompt_seconds": settings["s3gen_prompt_seconds"],
                "voice_encoder_seconds": reference_info["conditioning_duration_seconds"],
                "experimental": settings["experimental"],
            },
            "generation_parameters": generation_parameters,
            "base_seed": base_seed_value,
            "seed_policy": "base_seed_plus_chunk_index" if base_seed_value >= 0 else "random_seed_per_chunk_saved_in_manifest",
            "chunk_max_characters": int(max_chunk_chars),
            "speak_headings": bool(speak_headings),
            "pause_settings": plan["pause_settings"],
            "sample_rate": int(model.sr),
        }

        job_dir, manifest, resumed = longform.create_or_resume_job(
            JOBS_ROOT,
            requested_job_id or None,
            source_text=source_text,
            source_name=source_name,
            plan=plan,
            generation_config=generation_config,
            base_seed=base_seed_value,
        )
        job_id = manifest["job_id"]
        requested_rerenders = [
            item.strip().upper()
            for item in (rerender_chunk_ids or "").split(",")
            if item.strip()
        ]
        rerendered_ids = longform.force_rerender_chunks(
            job_dir,
            manifest,
            requested_rerenders,
            new_random_seed=bool(reroll_seed),
        )
        if rerendered_ids:
            longform.append_job_event(
                job_dir,
                "rerender_requested",
                "Selected chunks were queued for rerender.",
                details={"chunk_ids": rerendered_ids, "new_random_seed": bool(reroll_seed)},
            )
        longform.append_job_event(
            job_dir,
            "generation_started",
            f"{'Resuming' if resumed else 'Starting'} job {job_id} on {DEVICE}.",
            details={
                "chunk_count": len(manifest["chunks"]),
                "section_count": len(manifest["sections"]),
                "reference_sha256": reference_info["source_sha256"],
            },
        )
        manifest["status"] = "running"
        longform.write_manifest(job_dir, manifest)
        initial_status = (
            f"Job ID: {job_id}\n"
            f"{'Resuming' if resumed else 'Created'} | {len(manifest['chunks'])} chunks | "
            f"{len(manifest['sections'])} sections | {DEVICE}\n"
            "The manifest and log are checkpointed after every completed chunk."
        )
        manifest_path = str(job_dir / "manifest.json")
        log_path = str(job_dir / "job.log")
        yield initial_status, _batch_log_view(job_dir), None, None, manifest_path, log_path, job_id, None

        total = len(manifest["chunks"])
        finished_before = sum(
            item.get("status") == "complete" and longform.verify_chunk_wav(job_dir, item)
            for item in manifest["chunks"]
        )

        with MODEL_LOCK:
            progress(0, desc="Preparing narrator conditioning...")
            _prepare_batch_conditionals(
                normalized_reference,
                settings,
                float(exaggeration),
            )
            longform.append_job_event(
                job_dir,
                "reference_prepared",
                "Prepared narrator conditioning for this batch.",
                details={
                    "conditioning_profile": conditioning_profile,
                    "reference_duration_seconds": reference_info["conditioning_duration_seconds"],
                },
            )
            for chunk in manifest["chunks"]:
                if chunk.get("status") == "complete" and longform.verify_chunk_wav(job_dir, chunk):
                    continue

                chunk_id = chunk["chunk_id"]
                seed = int(chunk["seed"])
                chunk_started = time.perf_counter()
                chunk["status"] = "generating"
                chunk.pop("error", None)
                longform.write_manifest(job_dir, manifest)
                longform.append_job_event(
                    job_dir,
                    "chunk_started",
                    f"Generating {chunk_id} ({chunk['section_title']}, paragraph {chunk['paragraph_index']}).",
                    details={
                        "chunk_index": chunk["chunk_index"],
                        "seed": seed,
                        "text_sha256": chunk["text_sha256"],
                        "characters": len(chunk["text"]),
                    },
                )
                progress(
                    min(0.99, (finished_before + 0.25) / max(1, total)),
                    desc=f"Generating {chunk_id} ({finished_before}/{total} complete)",
                )
                try:
                    torch.manual_seed(seed)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed(seed)
                        torch.cuda.manual_seed_all(seed)
                    generated = model.generate(
                        chunk["text"],
                        language_id=language_id,
                        **generation_parameters,
                    )
                    waveform = generated.detach().float().cpu().numpy().squeeze()
                    if waveform.ndim != 1 or waveform.size == 0 or not np.isfinite(waveform).all():
                        raise RuntimeError("Model returned an empty, non-mono, or non-finite waveform.")
                    peak_before_limiter = float(np.max(np.abs(waveform)))
                    limiter_gain = 1.0
                    if peak_before_limiter > 0.98:
                        limiter_gain = 0.98 / peak_before_limiter
                        waveform *= limiter_gain

                    destination = job_dir / chunk["wav_relpath"]
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    temporary = destination.with_name(destination.stem + ".partial.wav")
                    sf.write(
                        str(temporary),
                        waveform.astype(np.float32, copy=False),
                        int(model.sr),
                        format="WAV",
                        subtype="PCM_16",
                    )
                    os.replace(temporary, destination)
                    duration = waveform.size / int(model.sr)
                    chunk.update({
                        "status": "complete",
                        "wav_sha256": longform.sha256_file(destination),
                        "duration_seconds": round(duration, 3),
                        "elapsed_seconds": round(time.perf_counter() - chunk_started, 3),
                        "peak_before_limiter": round(peak_before_limiter, 6),
                        "limiter_gain": round(limiter_gain, 6),
                        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
                        "device": DEVICE,
                    })
                    manifest["completed_chunks"] = sum(
                        item.get("status") == "complete" for item in manifest["chunks"]
                    )
                    longform.write_manifest(job_dir, manifest)
                    finished_before += 1
                    longform.append_job_event(
                        job_dir,
                        "chunk_complete",
                        f"Saved {chunk_id}: {duration:.2f}s audio.",
                        details={
                            "chunk_index": chunk["chunk_index"],
                            "seed": seed,
                            "wav_sha256": chunk["wav_sha256"],
                            "elapsed_seconds": chunk["elapsed_seconds"],
                        },
                    )
                    progress(
                        finished_before / max(1, total),
                        desc=f"Completed {finished_before}/{total} chunks",
                    )
                    current_status = (
                        f"Job ID: {job_id}\n"
                        f"Completed {finished_before}/{total} chunks. "
                        f"Last: {chunk_id}, {duration:.1f}s, seed {seed}.\n"
                        "Audio is checkpointed. A runtime interruption can be resumed with the same job ID and settings."
                    )
                    yield current_status, _batch_log_view(job_dir), None, None, manifest_path, log_path, job_id, None
                except Exception as exc:
                    chunk["status"] = "failed"
                    chunk["error"] = f"{type(exc).__name__}: {exc}"
                    manifest.setdefault("errors", []).append({
                        "chunk_id": chunk_id,
                        "error": chunk["error"],
                        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    })
                    manifest["status"] = "paused_after_error"
                    longform.write_manifest(job_dir, manifest)
                    longform.append_job_event(
                        job_dir,
                        "chunk_failed",
                        f"{chunk_id} failed: {type(exc).__name__}: {exc}",
                        level="ERROR",
                        details={"chunk_index": chunk["chunk_index"], "seed": seed},
                    )
                    yield (
                        f"Job ID: {job_id}\nStopped at {chunk_id}. {type(exc).__name__}: {exc}\n"
                        "Completed chunks are preserved. Fix the cause and resume with the same job ID/settings.",
                        _batch_log_view(job_dir),
                        None,
                        None,
                        manifest_path,
                        log_path,
                        job_id,
                        None,
                    )
                    return

        manifest = longform.read_manifest(job_dir)
        progress(0.995, desc="Assembling section and full-book WAVs...")
        artifacts = longform.assemble_job_masters(job_dir, manifest, sample_rate=int(model.sr))
        qa_report = longform.build_audio_qa_report(job_dir, manifest, sample_rate=int(model.sr))
        bundle = longform.create_job_bundle(job_dir)
        append_app_log(
            "audiobook_job_complete",
            f"Completed audiobook job {job_id}.",
            details={
                "job_id": job_id,
                "chunks": len(manifest["chunks"]),
                "sections": len(manifest["sections"]),
                "duration_seconds": artifacts["whole_book_master"]["audio"]["duration_seconds"],
            },
        )
        progress(1.0, desc="Audiobook masters and delivery bundle ready.")
        final_status = (
            f"COMPLETE: {job_id}\n"
            f"{len(manifest['chunks'])} chunks across {len(manifest['sections'])} sections.\n"
            f"Whole-book duration: {artifacts['whole_book_master']['audio']['duration_seconds'] / 60:.1f} minutes.\n"
            f"Technical QA: {qa_report['flagged_chunk_count']} chunk(s) flagged for listening review. "
            "See the QA report; automated checks cannot judge accent, pronunciation, or prosody. "
            "The bundle contains the source text, manifest, logs, chunk WAVs, section WAVs, QA report, and whole-book master."
        )
        yield (
            final_status,
            _batch_log_view(job_dir),
            str(job_dir / "whole_book_master.wav"),
            str(bundle),
            str(job_dir / "manifest.json"),
            str(job_dir / "job.log"),
            job_id,
            str(job_dir / "qa_report.json"),
        )
    except Exception as exc:
        if job_dir is not None:
            try:
                longform.append_job_event(
                    job_dir,
                    "job_failed",
                    f"{type(exc).__name__}: {exc}",
                    level="ERROR",
                )
                current_manifest = longform.read_manifest(job_dir)
                current_manifest["status"] = "failed"
                longform.write_manifest(job_dir, current_manifest)
            except Exception:
                pass
            log_text = _batch_log_view(job_dir)
            manifest_file = str(job_dir / "manifest.json")
            log_file = str(job_dir / "job.log")
            job_id_for_output = job_dir.name
            job_prefix = f"Job ID: {job_dir.name}\n"
        else:
            log_text = "No job log was created."
            manifest_file = None
            log_file = None
            job_id_for_output = requested_job_id or None
            job_prefix = ""
        yield (
            f"{job_prefix}Audiobook job failed: {type(exc).__name__}: {exc}",
            log_text,
            None,
            None,
            manifest_file,
            log_file,
            job_id_for_output,
            None,
        )

def load_activity_logs(job_id: str | None):
    candidate = (job_id or "").strip()
    if candidate:
        try:
            resolved = longform.safe_job_id(candidate)
            root = JOBS_ROOT / resolved
            if not root.is_dir():
                return f"No audiobook job named {resolved} was found.", None, None
            log_path = root / "job.log"
            manifest_path = root / "manifest.json"
            return (
                longform.read_log_tail(log_path, max_lines=250),
                str(manifest_path) if manifest_path.is_file() else None,
                str(log_path) if log_path.is_file() else None,
            )
        except Exception as exc:
            return f"Log lookup failed: {type(exc).__name__}: {exc}", None, None
    return (
        longform.read_log_tail(APP_LOG_PATH, max_lines=250),
        None,
        str(APP_LOG_PATH) if APP_LOG_PATH.is_file() else None,
    )

print("Long-form audiobook helpers loaded.")
print("Audiobook job root:", JOBS_ROOT)
print("App activity log:", APP_LOG_PATH)


## 7. Launch the Gradio web UI

The app has three work areas: quick one-off TTS, resumable long-form audiobook jobs, and activity/job logs. The temporary Gradio share URL has no login authentication. Anyone who obtains it can use the interface, so keep it private and stop the server when finished.

For long books, upload the reviewed UTF-8 manuscript as a .txt or .md file, preview the section/chunk plan, then confirm approval before starting. The batch job persists to Drive when Drive persistence is enabled; otherwise it lives in temporary Colab storage.

In [ ]:
REFERENCE_MODE_CHOICES = [
    "Automatic (English/Hindi demo; English fallback)",
    "Built-in English demo voice",
    "Built-in Hindi demo voice",
    "Upload/record my own voice",
]

with gr.Blocks(
    title="Chatterbox Multilingual V3 TTS and Audiobook Studio",
    analytics_enabled=False,
) as demo:
    gr.Markdown(
        """
        # Chatterbox Multilingual V3
        ### Voice playground and resumable audiobook studio

        Quick TTS is for auditions and short passages. Audiobook Jobs is for a reviewed
        manuscript: it saves every chunk, can resume after a Colab interruption, and assembles
        section masters and a whole-book WAV. Logs are available in the third tab.
        """
    )

    with gr.Tabs():
        with gr.Tab("Quick TTS"):
            with gr.Row():
                with gr.Column(scale=3):
                    text_input = gr.Textbox(
                        label="Text to speak",
                        placeholder="Type or paste your text here...",
                        lines=12,
                        max_lines=24,
                    )
                    language_input = gr.Dropdown(
                        choices=LANGUAGE_CHOICES,
                        value="en",
                        label="Text language",
                        interactive=True,
                    )
                    reference_mode_input = gr.Radio(
                        choices=REFERENCE_MODE_CHOICES,
                        value=REFERENCE_MODE_CHOICES[0],
                        label="Voice reference",
                    )
                    reference_audio_input = gr.Audio(
                        label="Upload or record a voice reference (used only in upload mode)",
                        sources=["upload", "microphone"],
                        type="filepath",
                        format="wav",
                    )
                    conditioning_profile_input = gr.Dropdown(
                        choices=list(CONDITIONING_PROFILES),
                        value="Standard (T3 6s / S3Gen 10s)",
                        label="Reference conditioning profile",
                        interactive=True,
                    )
                    gr.Markdown(
                        "Standard is the upstream default. The longer VoiceEncoder-sample profile "
                        "uses up to 30s for the T3 VoiceEncoder identity embedding but keeps the "
                        "original 6s/10s prompt windows. The extended profile experiments with a "
                        "10s T3 prompt and 15s S3Gen prompt; this can increase memory/time and may "
                        "reduce quality because it exceeds the model's usual conditioning lengths."
                    )
                    gr.Markdown(
                        "Tip: a clean 8–10 second sample of one speaker works well. "
                        "For languages other than English or Hindi, upload a reference in the target language."
                    )

                with gr.Column(scale=2):
                    with gr.Accordion("Generation settings", open=True):
                        exaggeration_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                            label="Exaggeration",
                        )
                        cfg_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                            label="CFG weight",
                        )
                        temperature_input = gr.Slider(
                            minimum=0.05, maximum=2.0, value=0.7, step=0.05,
                            label="Temperature",
                        )
                        repetition_input = gr.Slider(
                            minimum=1.0, maximum=2.0, value=1.2, step=0.05,
                            label="Repetition penalty",
                        )
                        min_p_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.05, step=0.01,
                            label="Min-p",
                        )
                        top_p_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.95, step=0.01,
                            label="Top-p",
                        )
                        seed_input = gr.Number(
                            value=-1,
                            precision=0,
                            label="Seed (-1 = random)",
                        )
                    gr.Markdown(
                        "**Repetition-heavy passages:** try **Temperature 0.55** and "
                        "**Repetition penalty 1.35 together**. This stopped one reproduced "
                        "repetition issue; it is a targeted starting point, not a guarantee."
                    )
                    generate_button = gr.Button("Generate speech", variant="primary")

            audio_output = gr.Audio(label="Generated audio", type="filepath")
            file_output = gr.File(label="Download WAV")
            status_output = gr.Textbox(label="Status", interactive=False, lines=3)
            generate_button.click(
                fn=generate_speech,
                inputs=[
                    text_input,
                    language_input,
                    reference_mode_input,
                    reference_audio_input,
                    conditioning_profile_input,
                    exaggeration_input,
                    cfg_input,
                    temperature_input,
                    repetition_input,
                    min_p_input,
                    top_p_input,
                    seed_input,
                ],
                outputs=[audio_output, file_output, status_output],
                api_name=False,
            )

        with gr.Tab("Audiobook Jobs"):
            gr.Markdown(
                """
                ### Full-length, checkpointed synthesis

                Upload one approved UTF-8 .txt or .md manuscript, or paste a shorter text.
                Common headings such as Chapter 1, Preface, Introduction, Notes, and Appendix
                become separate output sections. Headings are spoken by default. Preview the
                plan before generating.

                Each chunk is saved separately with its own persisted seed and SHA-256. If Colab
                disconnects, reopen this notebook, load the same voice reference, enter the same
                job ID and settings, and run again. Verified chunks are skipped; missing or
                damaged chunks are regenerated. Drive persistence is strongly recommended.
                """
            )
            with gr.Row():
                with gr.Column(scale=3):
                    batch_file_input = gr.File(
                        label="Approved manuscript file (.txt or .md)",
                        file_types=[".txt", ".md"],
                        type="filepath",
                    )
                    batch_text_input = gr.Textbox(
                        label="Or paste manuscript text (leave blank when uploading a file)",
                        lines=10,
                        max_lines=20,
                    )
                    batch_job_id_input = gr.Textbox(
                        label="Job ID (leave blank to create a new job; reuse it to resume)",
                        placeholder="Example: buddhist-yoga-chapter-01",
                    )
                    batch_rerender_ids_input = gr.Textbox(
                        label="Force-rerender specific chunks (optional)",
                        placeholder="Example: C00007, C00012",
                    )
                    batch_reroll_seed_input = gr.Checkbox(
                        value=True,
                        label="Assign a fresh random seed to rerendered chunks",
                    )
                    speak_headings_input = gr.Checkbox(
                        value=True,
                        label="Speak detected section headings",
                    )
                    batch_chunk_chars_input = gr.Slider(
                        minimum=160, maximum=520, value=320, step=20,
                        label="Maximum characters per TTS chunk",
                    )
                    with gr.Accordion("Pauses at joins", open=False):
                        batch_chunk_pause_input = gr.Slider(
                            minimum=0.0, maximum=0.8, value=0.18, step=0.02,
                            label="Within-paragraph chunk pause (seconds)",
                        )
                        batch_paragraph_pause_input = gr.Slider(
                            minimum=0.0, maximum=1.5, value=0.45, step=0.05,
                            label="Paragraph pause (seconds)",
                        )
                        batch_section_pause_input = gr.Slider(
                            minimum=0.0, maximum=3.0, value=1.0, step=0.1,
                            label="Section/chapter pause in whole-book master (seconds)",
                        )
                    preview_button = gr.Button("Preview section and chunk plan")
                    plan_preview_output = gr.Textbox(
                        label="Plan preview (no audio generated)",
                        lines=14,
                        max_lines=24,
                        interactive=False,
                    )

                with gr.Column(scale=2):
                    batch_language_input = gr.Dropdown(
                        choices=LANGUAGE_CHOICES,
                        value="en",
                        label="Manuscript language",
                        interactive=True,
                    )
                    batch_reference_mode_input = gr.Radio(
                        choices=REFERENCE_MODE_CHOICES,
                        value="Upload/record my own voice",
                        label="Narrator voice reference",
                    )
                    batch_reference_audio_input = gr.Audio(
                        label="Upload/record narrator reference",
                        sources=["upload", "microphone"],
                        type="filepath",
                        format="wav",
                    )
                    batch_conditioning_profile_input = gr.Dropdown(
                        choices=list(CONDITIONING_PROFILES),
                        value="Standard (T3 6s / S3Gen 10s)",
                        label="Reference conditioning profile",
                        interactive=True,
                    )
                    with gr.Accordion("Narration generation settings", open=True):
                        batch_exaggeration_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                            label="Exaggeration",
                        )
                        batch_cfg_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                            label="CFG weight",
                        )
                        batch_temperature_input = gr.Slider(
                            minimum=0.05, maximum=2.0, value=0.55, step=0.05,
                            label="Temperature",
                        )
                        batch_repetition_input = gr.Slider(
                            minimum=1.0, maximum=2.0, value=1.35, step=0.05,
                            label="Repetition penalty",
                        )
                        batch_min_p_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.05, step=0.01,
                            label="Min-p",
                        )
                        batch_top_p_input = gr.Slider(
                            minimum=0.0, maximum=1.0, value=0.95, step=0.01,
                            label="Top-p",
                        )
                        batch_seed_input = gr.Number(
                            value=-1,
                            precision=0,
                            label="Base seed (-1 = save a random seed per chunk)",
                        )
            approval_checkbox = gr.Checkbox(
                value=False,
                label="I have reviewed and approved this exact reading copy, including headings and spoken-form spellings, and authorize this batch.",
            )
            run_batch_button = gr.Button("Generate / resume audiobook job", variant="primary")
            batch_status_output = gr.Textbox(label="Job status", lines=4, interactive=False)
            batch_live_log_output = gr.Textbox(
                label="Live job log (updates after each chunk)",
                lines=12,
                max_lines=20,
                interactive=False,
            )
            with gr.Row():
                batch_master_output = gr.File(label="Whole-book master WAV")
                batch_bundle_output = gr.File(label="Complete delivery bundle ZIP")
            with gr.Row():
                batch_manifest_output = gr.File(label="Job manifest JSON")
                batch_log_file_output = gr.File(label="Job log TXT")
                batch_qa_output = gr.File(label="Technical audio QA report JSON")

            preview_button.click(
                fn=preview_audiobook_plan,
                inputs=[
                    batch_file_input,
                    batch_text_input,
                    batch_chunk_chars_input,
                    speak_headings_input,
                    batch_chunk_pause_input,
                    batch_paragraph_pause_input,
                    batch_section_pause_input,
                ],
                outputs=[plan_preview_output],
                api_name=False,
            )
            run_batch_button.click(
                fn=run_audiobook_job,
                inputs=[
                    batch_file_input,
                    batch_text_input,
                    batch_job_id_input,
                    batch_rerender_ids_input,
                    batch_reroll_seed_input,
                    speak_headings_input,
                    batch_chunk_chars_input,
                    batch_chunk_pause_input,
                    batch_paragraph_pause_input,
                    batch_section_pause_input,
                    approval_checkbox,
                    batch_language_input,
                    batch_reference_mode_input,
                    batch_reference_audio_input,
                    batch_conditioning_profile_input,
                    batch_exaggeration_input,
                    batch_cfg_input,
                    batch_temperature_input,
                    batch_repetition_input,
                    batch_min_p_input,
                    batch_top_p_input,
                    batch_seed_input,
                ],
                outputs=[
                    batch_status_output,
                    batch_live_log_output,
                    batch_master_output,
                    batch_bundle_output,
                    batch_manifest_output,
                    batch_log_file_output,
                    batch_job_id_input,
                    batch_qa_output,
                ],
                api_name=False,
            )

        with gr.Tab("Logs and Recovery"):
            gr.Markdown(
                """
                ### Activity, logs, and recovery records

                Enter an audiobook Job ID to inspect its log and manifest. Leave it blank to
                inspect the quick-TTS app activity log. The job log records chunk starts,
                completions, failures, seeds, elapsed times, and output hashes. It does not
                record your narrator audio. The job manifest contains the manuscript text in
                chunk records, so keep it private.
                """
            )
            logs_job_id_input = gr.Textbox(
                label="Audiobook Job ID (blank = quick-TTS activity log)",
                placeholder="Paste the Job ID shown in the Audiobook Jobs tab",
            )
            refresh_logs_button = gr.Button("Refresh logs")
            activity_log_output = gr.Textbox(
                label="Recent activity",
                lines=18,
                max_lines=28,
                interactive=False,
            )
            with gr.Row():
                inspected_manifest_output = gr.File(label="Download inspected manifest")
                inspected_log_output = gr.File(label="Download inspected log")
            refresh_logs_button.click(
                fn=load_activity_logs,
                inputs=[logs_job_id_input],
                outputs=[activity_log_output, inspected_manifest_output, inspected_log_output],
                api_name=False,
            )


demo.queue()
print("Starting the Gradio app without login authentication...")
print("After the next cell output appears, open the Gradio public URL.")
demo.launch(
    share=True,
    server_name="0.0.0.0",
    debug=True,
    show_error=True,
    max_file_size="25mb",
    allowed_paths=[str(LOCAL_OUTPUT_DIR)] + ([str(DRIVE_OUTPUT_DIR)] if DRIVE_OUTPUT_DIR is not None else []),
    enable_monitoring=False,
)

## Usage notes and troubleshooting

- **Google Drive is optional:** the notebook does not request Drive access when `USE_GOOGLE_DRIVE = False`. This is the default. Generated WAVs and JSON metadata stay in the Colab VM and can be downloaded through Gradio, but are deleted when the runtime is recycled.
- **Enable Drive persistence:** set `USE_GOOGLE_DRIVE = True` in the setup cell before running it, then execute the notebook from the top. Authorize Google Drive when prompted. The V3 model snapshot and generated WAV/JSON files are then persisted under `MyDrive/Chatterbox Playground`.
- **Model download:** without Drive, the model is downloaded to the Colab VM each time you start a fresh runtime. With Drive enabled, the notebook reuses its verified model snapshot when available.
- **No GPU:** the notebook loads on CPU if CUDA is unavailable, but synthesis will be considerably slower.
- **Gradio link:** the share link is temporary and stops working when the Colab runtime disconnects. Relaunch the final cell to create a new one.
- **Public Gradio link:** authentication is disabled. Anyone with the temporary share URL can use the app. Keep the URL private and stop the Gradio server when finished.
- **Mixed scripts / repeated phrases:** the selected language ID is applied to every character in the input. For English narration, do not insert an isolated Devanagari word unless the English tokenizer actually supports it. The app now records unknown `[UNK]` text tokens in the JSON metadata and warns in the status field. If a comma-separated list loops, try semicolons between items and rerender with a fixed seed; this changes pauses, not the words.
- **Voice reference:** use one speaker, clean speech, little background noise, and preferably 8–10 seconds in the standard profile. The standard profile uses 6 seconds for the T3 speech prompt and 10 seconds for S3Gen. The longer VoiceEncoder-sample profile can feed up to 30 seconds to the T3 VoiceEncoder identity embedding. The extended prompt profile experiments with 10 seconds for T3 and 15 seconds for S3Gen. Those longer prompt windows are not the upstream defaults and should be judged by listening; more audio is not automatically better. Your original upload and normalized temporary copy stay in the active Colab runtime and are not copied to Drive.
- **Repeated phrases / list loops:** in one reproduced case, Temperature **0.55** and Repetition penalty **1.35 together** stopped the repetition; either setting alone did not. Try the pair together while leaving the other controls unchanged. This is an empirically useful preset for repetition-prone passages, not a universal guarantee. The app keeps its general-purpose defaults unchanged.
- **Full audiobook jobs:** use the Audiobook Jobs tab with a reviewed UTF-8 manuscript file. Preview the section/chunk plan first. The job saves the original source, one WAV per chunk, a manifest, logs, per-section masters, a streamed whole-book master, and a delivery ZIP. It checkpoints after each chunk and resumes only when the source text, chunk plan, voice reference, and generation settings match.
- **Resume after Colab reset:** keep USE_GOOGLE_DRIVE = True before running setup. Reopen the notebook, run setup/model cells, upload the exact same narrator reference, enter the original Job ID, restore the same chunk/pause/generation controls, check the approval box, and run again. Completed WAVs are hash-verified and skipped. Do not change settings mid-job; create a new Job ID for a deliberate new render.
- **Seeds in batch jobs:** with base seed -1, the notebook assigns and saves an independent random seed per chunk. Those seeds are reused on resume. With a non-negative base seed, each chunk gets base seed + its zero-based chunk offset. The manifest records every seed.
- **Section detection:** Markdown headings and common headings such as Chapter, Preface, Introduction, Notes, and Appendix start new sections. Headings are spoken by default; inspect the preview and toggle this before synthesis. The original source file is retained unchanged in the job folder and ZIP.
- **Audio masters:** section WAVs and the whole-book master are 24 kHz mono PCM16. Assembly streams from chunk files instead of loading the complete book into memory. The ZIP includes chunk audio, masters, source, manifest, and logs. Keep it private; the manifest contains your manuscript text.
- **Logs and recovery:** use Logs and Recovery to inspect the quick-TTS event history or a specific audiobook Job ID. The live log updates after each chunk and is saved both as readable text and JSONL events.
- **Fix a bad chunk:** after listening to the book, enter its chunk ID (for example, C00007) in the force-rerender field. The previous WAV is preserved in the job's revisions folder and ZIP; a fresh random seed is assigned by default. Only selected chunks are rerendered, then section and whole-book masters are rebuilt.
- **Technical QA report:** the job writes qa_report.json and includes it in the delivery ZIP. It checks WAV hashes/format, unusual duration or estimated speaking rate, sentence-internal splits, and limiter engagement. Flags are listening prompts only; the report cannot detect accent, pronunciation, repetition, prosody, or omitted meaning.
- **Full-book safeguard:** the per-job checkbox is a user confirmation only. It does not replace manuscript approval or unlock the separate document-first audiobook workflow's approval gate.
- **Long passages in Quick TTS:** Quick TTS still uses 420-character chunks and is intended for auditions, not whole-book processing.
- **Output privacy:** quick-TTS metadata includes the entered text and settings. Audiobook job manifests and bundles include the manuscript text, settings, seeds, logs, and audio. Keep the Drive folder or downloaded files private.
- **Model mismatch:** if the V3 selector is missing or the notebook imports an old package, restart the Colab runtime and rerun from the top. The notebook intentionally uses the pinned upstream source commit rather than the old PyPI wheel.
- **This is not book synthesis:** nothing in this notebook changes the manuscript, pronunciation decisions, roadmap, or full-book synthesis lock.

Useful links: [Chatterbox source](https://github.com/resemble-ai/chatterbox) · [Gradio Audio component](https://gradio.app/docs/audio) · [Gradio Blocks](https://gradio.app/docs/gradio/blocks).